In [14]:
from pprint import pprint

from pathlib import Path

from hlwdetector.runner import ExperimentRunner
from hlwdetector.visualization.confusion_matrix import ConfusionMatrixVisualizer
from hlwdetector.visualization.metrics_comparator import MetricsComparator

# Prepare Dataset
**Prerequesites**
1. Directory containing annotated video dataset (mp4 files)
2. COCO annotations JSON where image filenames map each frame to the originating video, for example: ```IMG_0070_frame_000000.png```
3. Train/val/test split defined as a list of video file names for each split in split.json
```
{
    "train": [],
    "val": [],
    "test" []
}
```
See ````prepare_yolo_dataset.ipynb```` for how to produce split.json

## Extract frames

In [ ]:
#video_dir = Path("data/h23/videos")  # Directory containing annotated video dataset
#split_json = Path("data/h23/split.json")  # COCO annotations JSON
#out_dir = Path("data/h23/images")  # Directory to extract split images

# Uncomment the below line to extract video frames into split directories

# extract_frames_by_split(split_json, video_dir, out_dir)

# Run experiment

In [ ]:
#config_yaml = "configs/yolo11_h23.yaml"
#config_yaml = "configs/yolo11_h23_resume.yaml"
config_yaml = "configs/yolo26_h23.yaml"
runner = ExperimentRunner(config_yaml)
pprint(runner.config)

In [ ]:
runner.run_pipeline()

In [ ]:
rtdetr_config = "configs/rtdetr_h23.yaml"
rtdetr_runner = ExperimentRunner(rtdetr_config)
rtdetr_runner.run_pipeline()

In [ ]:
runner.train()

In [ ]:
runner = ExperimentRunner.from_experiment_dir("outputs/yolo11_h23_20260416_083051")

In [ ]:
detections = runner.artifact_manager.load_detections()
conf_mat_viz = ConfusionMatrixVisualizer(
    config=runner.config,
    artifact_manager=runner.artifact_manager,
    dataset_manager=runner.dataset_manager
)
conf_mat_result = conf_mat_viz.compute(detections, confidence_threshold=0.5)
print(conf_mat_result)
conf_mat_viz.plot(conf_mat_result)

In [ ]:
tp_frames = conf_mat_viz.sample_frames(conf_mat_result, "tp", n=2, detections=detections)
tn_frames = conf_mat_viz.sample_frames(conf_mat_result, "tn", n=2, detections=detections)
fp_frames = conf_mat_viz.sample_frames(conf_mat_result, "fp", n=2, detections=detections)
fn_frames = conf_mat_viz.sample_frames(conf_mat_result, "fn", n=2, detections=detections)

In [20]:
dirs = ["outputs/yolo11_h23_20260416_083051", "outputs/yolo26_h23_20260416_085928"]
m_comparator = MetricsComparator.from_experiment_dirs(dirs)
df = m_comparator.to_dataframe()
display(df)


,accuracy,precision,recall,f1,mAP50,mAP50:0.95
experiment,,,,,,
yolo11_h23,None,0.453099,0.025000,0.047385,0.026980,0.013620
yolo26_h23,None,0.212713,0.041667,0.069684,0.036209,0.014981


In [ ]:
runner.evaluate()

In [ ]:
runner.predict()

In [ ]:
runner = ExperimentRunner.from_experiment_dir("outputs/yolo26_h23_20260415_210100")
runner.visualize_predictions()
runner.tracker.finish()

In [ ]:
#!jupyter nbconvert --to python run_experiments.ipynb